In [3]:
# from huggingface_hub import snapshot_download

# snapshot_download(
#     repo_id="malaysia-ai/Malaysian-STT", 
#     allow_patterns="*/*.parquet",
#     local_dir="./Malaysian-STT",
#     repo_type="dataset",
# )

In [4]:
import pandas as pd
import json
import os
import itertools
from glob import glob
from transformers import AutoTokenizer, AddedToken
from multiprocess import Pool

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)


def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

def new_path(f):
    f = f.replace('.mp3', '.glm4')
    splitted = f.split('/')
    base_folder = splitted[0] + '_glm4'
    splitted = '/'.join([base_folder] + splitted[1:])
    return splitted

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [5]:
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B')
extra = [
    AddedToken('<|endofspeech|>'), 
    AddedToken('<|whole|>'), 
    AddedToken('<|streaming|>'),
    AddedToken('<|segment|>'),
    AddedToken('<|word|>'),
    AddedToken('<|'),
    AddedToken('|>')
]
for i in range(16384):
    extra.append(AddedToken(f'<|s{i}|>'))
tokenizer.add_tokens(extra)

16391

In [6]:
files = glob('Malaysian-STT/*/*.parquet')
files

['Malaysian-STT/malaysian/train-00003-of-00004.parquet',
 'Malaysian-STT/malaysian/train-00000-of-00004.parquet',
 'Malaysian-STT/malaysian/train-00001-of-00004.parquet',
 'Malaysian-STT/malaysian/train-00002-of-00004.parquet',
 'Malaysian-STT/science_english/train-00006-of-00007.parquet',
 'Malaysian-STT/science_english/train-00000-of-00007.parquet',
 'Malaysian-STT/science_english/train-00001-of-00007.parquet',
 'Malaysian-STT/science_english/train-00004-of-00007.parquet',
 'Malaysian-STT/science_english/train-00003-of-00007.parquet',
 'Malaysian-STT/science_english/train-00005-of-00007.parquet',
 'Malaysian-STT/science_english/train-00002-of-00007.parquet',
 'Malaysian-STT/imda/train-00001-of-00008.parquet',
 'Malaysian-STT/imda/train-00002-of-00008.parquet',
 'Malaysian-STT/imda/train-00006-of-00008.parquet',
 'Malaysian-STT/imda/train-00005-of-00008.parquet',
 'Malaysian-STT/imda/train-00000-of-00008.parquet',
 'Malaysian-STT/imda/train-00003-of-00008.parquet',
 'Malaysian-STT/imd

In [7]:
from tqdm import tqdm

def loop(files):
    files, _ = files
    for file in files:
        folder = file.replace('.parquet', '').replace('/', '_') + '_done'
        os.makedirs(folder, exist_ok = True)
        df = pd.read_parquet(file)
        prefix_file = os.path.split(file)[1].replace('.parquet', '')
        for i in tqdm(range(len(df))):
        
            filename_done = f'{folder}/{i}.json'
            if os.path.exists(filename_done):
                try:
                    with open(filename_done) as fopen:
                        json.load(fopen)
                    continue
                except:
                    pass
        
            mode = df['mode'].iloc[i]
            level = df['level'].iloc[i]
            prefix = f'<|{mode}|><|{level}|>'
            
            tokens = []
            temp = []
        
            not_found = False
            for no, f in enumerate(df['audio_filenames'].iloc[i]):
                new_f = new_path(f)
                if not os.path.exists(new_f):
                    if len(temp):
                        tokens.append(''.join(temp))
                        temp = []
                        not_found = True
                    break
            
                t = df['texts'].iloc[i][no]
                try:
                    with open(new_f) as fopen:
                        token = json.load(fopen)
                except:
                    break
                token = ''.join([f'<|s{t}|>' for t in token]) + '<|endofspeech|>'
                token = token + t + '<|endoftext|>'
                temp.append(token)
            
            if len(temp):
                tokens.append(''.join(temp))
        
            for i in range(len(tokens)):
                tokens[i] = prefix + tokens[i]
        
            tokens = ''.join(tokens)
        
            with open(filename_done, 'w') as fopen:
                json.dump(tokens, fopen)

In [8]:
len(files)

54

In [ ]:
multiprocessing(files, loop, cores = 50, returned = False)

  2%|▏         | 25997/1344354 [00:17<09:20, 2351.01it/s]